### Generic COPY INTO Framework

### Design Goal
The Bronze layer was implemented using a metadata-driven ingestion framework to ensure:

- scalable ingestion for multiple datasets  
- minimal duplication of code  
- rerun-safe incremental processing  
- consistent audit and lineage metadata  

---

### Components

#### 1) Generic ingestion notebook
- accepts table parameters via Databricks widgets  
- reads table-specific configuration from YAML  
- runs `COPY INTO` to ingest raw files into Delta Bronze tables  

#### 2) Global utilities notebook
- shared functions for config loading, path generation, and DDL generation  
- reusable across ingestion notebooks  

#### 3) YAML ingestion config
- defines table schema, file format, source location, and `COPY INTO` options  
- allows adding folder-level lineage (`source`) per table  

---

### Bronze Table Standards
All Bronze tables contain:

- raw columns stored as `STRING`  
- audit metadata:
  - `loaded_at`
  - `updated_at`
  - `load_dt`
  - `source_file`
- optional lineage column:
  - `source` (folder-level)


In [0]:
# ============================================================
# bronze_copy_into_generic
# One notebook to ingest ANY COPY INTO Bronze table
# ============================================================

# -------------------------
# Job Parameters (Widgets)
# -------------------------
dbutils.widgets.text("catalog", "coffee")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("table_name", " ")
dbutils.widgets.text("raw_volume", " ")



catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
table_name = dbutils.widgets.get("table_name")
raw_volume = dbutils.widgets.get("raw_volume")


if not table_name:
    raise ValueError("table_name job parameter is required")



In [0]:
%run ../common/Globals
#running the global notebook to acces the function

In [0]:



# -------------------------
# Load Config
# -------------------------
config_path = "/Workspace/Users/mkmanpreetkaur0016@gmail.com/databricks-coffee-data-pipeline/src/resources/bronze_config.yml"
cfg = load_config(config_path)

if "tables" not in cfg:
    raise ValueError("Invalid config: missing top-level 'tables' key")

if table_name not in cfg["tables"]:
    raise ValueError(f"Table '{table_name}' not found in config")

t = cfg["tables"][table_name]

# Validate ingestion type
if t.get("ingestion_type") != "copy_into":
    raise ValueError(
        f"Table '{table_name}' is not configured for COPY INTO. "
        f"Found ingestion_type={t.get('ingestion_type')}"
    )

# -------------------------
# Read table config values
# -------------------------
columns = t["columns"]
source_subfolder = t["source_subfolder"]

file_format = t.get("file_format", "CSV")
format_options = t.get("format_options", {"header": "true"})
copy_options = t.get("copy_options", {"mergeSchema": "false"})

add_source_column = t.get("add_source_column", False)
source_value = t.get("source_value", None)

if add_source_column and not source_value:
    raise ValueError(f"Table '{table_name}' requires source_value in config")

# -------------------------
# Build paths
# -------------------------
target_table = build_table_fqn(catalog, bronze_schema, table_name)
source_path = build_source_path(raw_volume, source_subfolder)

print("--------------------------------------------------")
print("Running COPY INTO Bronze ingestion")
print(f"Target table : {target_table}")
print(f"Source path  : {source_path}")
print(f"Add source?  : {add_source_column}")
print("--------------------------------------------------")

# -------------------------
# Step 1: Create table (all STRING)
# -------------------------
ddl_sql = build_create_table_sql(
    table_fqn=target_table,
    raw_columns=columns,
    add_source_column=add_source_column
)
spark.sql(ddl_sql)

# -------------------------
# Step 2: Build SELECT list
# -------------------------
audit_exprs = [
    "current_timestamp() AS loaded_at",
    "current_timestamp() AS updated_at",
    "current_date() AS load_dt",
    "_metadata.file_name AS source_file"
]

if add_source_column:
    audit_exprs.append(f"'{source_value}' AS source")

select_list = ",\n    ".join(columns + audit_exprs)

# -------------------------
# Step 3: Build options SQL
# -------------------------
format_opts_sql = ", ".join([f"'{k}' = '{v}'" for k, v in format_options.items()])
copy_opts_sql = ", ".join([f"'{k}' = '{v}'" for k, v in copy_options.items()])

# -------------------------
# Step 4: COPY INTO
# -------------------------
copy_sql = f"""
COPY INTO {target_table}
FROM (
  SELECT
    {select_list}
  FROM '{source_path}'
)
FILEFORMAT = {file_format}
FORMAT_OPTIONS ({format_opts_sql})
COPY_OPTIONS ({copy_opts_sql})
"""

spark.sql(copy_sql)

print(f"COPY INTO completed successfully for: {target_table}")
